In [45]:
import pandas as pd

In [46]:
df = pd.read_csv('data.csv')

In [47]:
df.head()

,Age,Gender,Education,Introversion Score,Sensing Score,Thinking Score,Judging Score,Interest,Personality
0,21.0,Female,1,5.89208,2.144395,7.32363,5.462224,Arts,ENTP
1,24.0,Female,1,2.48366,3.206188,8.06876,3.765012,Unknown,INTP
2,26.0,Female,1,7.02910,6.469302,4.16472,5.454442,Others,ESFP
3,30.0,Male,0,5.46525,4.179244,2.82487,5.080477,Sports,ENFJ
4,31.0,Female,0,3.59804,6.189259,5.31347,3.677984,Others,ISFP


In [48]:
df.shape

(43744, 9)

In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43744 entries, 0 to 43743
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 43744 non-null  float64
 1   Gender              43744 non-null  object 
 2   Education           43744 non-null  int64  
 3   Introversion Score  43744 non-null  float64
 4   Sensing Score       43744 non-null  float64
 5   Thinking Score      43744 non-null  float64
 6   Judging Score       43744 non-null  float64
 7   Interest            43744 non-null  object 
 8   Personality         43744 non-null  object 
dtypes: float64(5), int64(1), object(3)
memory usage: 3.0+ MB


In [50]:
df.describe()

,Age,Education,Introversion Score,Sensing Score,Thinking Score,Judging Score
count,43744.000000,43744.000000,43744.000000,43744.000000,43744.000000,43744.000000
mean,27.437203,0.229014,4.588349,5.780074,5.419131,5.391041
std,4.893805,0.420203,2.902628,1.241648,2.900785,1.442413
min,18.000000,0.000000,0.000150,0.000000,0.000320,0.000000
25%,24.000000,0.000000,2.067020,4.953340,2.895750,4.511842
50%,27.000000,0.000000,4.261680,6.162928,5.769870,5.771635
75%,30.000000,0.000000,7.085002,6.622978,7.923503,6.409583
max,52.000000,1.000000,9.999920,9.803837,9.999770,10.000000


In [51]:
df.isnull().sum()

Age                   0
Gender                0
Education             0
Introversion Score    0
Sensing Score         0
Thinking Score        0
Judging Score         0
Interest              0
Personality           0
dtype: int64

In [52]:
df = df.drop_duplicates()

In [53]:
df['Personality'].value_counts()

Personality
ENTP    2734
INTP    2734
ESFP    2734
ENFJ    2734
ISFP    2734
ISFJ    2734
ESTJ    2734
INFP    2734
ESTP    2734
ENFP    2734
INTJ    2734
ESFJ    2734
INFJ    2734
ISTP    2734
ENTJ    2734
ISTJ    1706
Name: count, dtype: int64

In [54]:
df['Gender'] = df['Gender'].map({'Male':1 , 'Female': 0})
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42716 entries, 0 to 43743
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Age                 42716 non-null  float64
 1   Gender              42716 non-null  int64  
 2   Education           42716 non-null  int64  
 3   Introversion Score  42716 non-null  float64
 4   Sensing Score       42716 non-null  float64
 5   Thinking Score      42716 non-null  float64
 6   Judging Score       42716 non-null  float64
 7   Interest            42716 non-null  object 
 8   Personality         42716 non-null  object 
dtypes: float64(5), int64(2), object(2)
memory usage: 3.3+ MB


In [55]:
df.head()

,Age,Gender,Education,Introversion Score,Sensing Score,Thinking Score,Judging Score,Interest,Personality
0,21.0,0,1,5.89208,2.144395,7.32363,5.462224,Arts,ENTP
1,24.0,0,1,2.48366,3.206188,8.06876,3.765012,Unknown,INTP
2,26.0,0,1,7.02910,6.469302,4.16472,5.454442,Others,ESFP
3,30.0,1,0,5.46525,4.179244,2.82487,5.080477,Sports,ENFJ
4,31.0,0,0,3.59804,6.189259,5.31347,3.677984,Others,ISFP


In [56]:
X = df.drop(['Personality'],axis=1)
y = df['Personality']

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [58]:
X_train.shape

(29901, 8)

In [40]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

scale_col = df.select_dtypes(include='float').columns
cat_col = ['Interest']

preprocess = ColumnTransformer(transformers=[
    ('ss', StandardScaler(), scale_col),
    ('encode', OneHotEncoder(handle_unknown='ignore', dtype=int), cat_col)
], remainder='passthrough')



In [41]:
X_train_pre = preprocess.fit_transform(X_train)
X_test_pre = preprocess.transform(X_test)

In [43]:
X_train_pre.shape

(29901, 12)

In [59]:
preprocess.get_feature_names_out()

array(['ss__Age', 'ss__Introversion Score', 'ss__Sensing Score',
       'ss__Thinking Score', 'ss__Judging Score', 'encode__Interest_Arts',
       'encode__Interest_Others', 'encode__Interest_Sports',
       'encode__Interest_Technology', 'encode__Interest_Unknown',
       'remainder__Gender', 'remainder__Education'], dtype=object)

In [72]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
lr = LogisticRegression()
lr.fit(X_train_pre, y_train)
y_pred = lr.predict(X_test_pre)
y_pred_train = lr.predict(X_train_pre)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(precision_score(y_train, y_pred_train, average='weighted'))
print(recall_score(y_train, y_pred_train, average='weighted'))
print(f1_score(y_train, y_pred_train, average='weighted'))
print('test')
print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred, average='weighted'))
print(recall_score(y_test, y_pred, average='weighted'))
print(f1_score(y_test, y_pred, average='weighted'))

train
0.8487675997458279
0.8507152043500895
0.8487675997458279
0.8487444471858812
test
0.8487709715177526
0.8507843879796961
0.8487709715177526
0.8486469238210359


In [74]:
from sklearn.naive_bayes import GaussianNB

naive = GaussianNB()
naive.fit(X_train_pre, y_train)
y_pred = naive.predict(X_test_pre)
y_pred_train = naive.predict(X_train_pre)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(precision_score(y_train, y_pred_train, average='weighted', zero_division=0))
print(recall_score(y_train, y_pred_train, average='weighted'))
print(f1_score(y_train, y_pred_train, average='weighted'))
print('test')
print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred, average='weighted',zero_division=0))
print(recall_score(y_test, y_pred, average='weighted'))
print(f1_score(y_test, y_pred, average='weighted'))

train
0.6057656934550684
0.7189271228409754
0.6057656934550684
0.6306434248541285
test
0.603667577058135
0.7182420743408585
0.603667577058135
0.6269176275073729


In [75]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_pre, y_train)
y_pred = knn.predict(X_test_pre)
y_pred_train = knn.predict(X_train_pre)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(precision_score(y_train, y_pred_train, average='weighted', zero_division=0))
print(recall_score(y_train, y_pred_train, average='weighted'))
print(f1_score(y_train, y_pred_train, average='weighted'))
print('test')
print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred, average='weighted', zero_division=0))
print(recall_score(y_test, y_pred, average='weighted'))
print(f1_score(y_test, y_pred, average='weighted'))

train
0.838901708972944
0.8413088023604758
0.838901708972944
0.8383952277016898
test
0.743425673039407
0.7471026954598476
0.743425673039407
0.7416927243788577


In [76]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier()
param = [{
    'n_estimators':[51,101,201],
    'max_depth':[2,3,4],
    'min_samples_leaf':[3,5,7]
}]
grid = GridSearchCV(estimator=rf, param_grid=param, n_jobs=3, cv=5)
grid.fit(X_train_pre, y_train)
y_pred = grid.predict(X_test_pre)
y_pred_train = grid.predict(X_train_pre)

print('train')
print(accuracy_score(y_train, y_pred_train))
print(precision_score(y_train, y_pred_train, average='weighted', zero_division=0))
print(recall_score(y_train, y_pred_train, average='weighted', zero_division=0))
print(f1_score(y_train, y_pred_train, average='weighted', zero_division=0))
print('test')
print(accuracy_score(y_test, y_pred))
print(precision_score(y_test, y_pred, average='weighted',zero_division=0))
print(recall_score(y_test, y_pred, average='weighted'))
print(f1_score(y_test, y_pred, average='weighted'))

train
0.8230159526437243
0.8324167139674998
0.8230159526437243
0.8236416125396677
test
0.8224736636753804
0.8323832849933925
0.8224736636753804
0.8230191853933657
